# PKG Geo — Phase A2: does the geographic story survive?

Four checks, in priority order. Any one of them can invalidate the Phase A summary,
so they run before any further analysis and before the edge pass is scoped.

| | Check | What it could kill |
|---|---|---|
| **1** | **Business vs consumer** | If the Florida inflow is households, this is a Retail/Wealth finding, not a TM one |
| **2** | **NAICS shift-share** | If metro net flow is industry mix, "geography" is a label on an industry effect |
| **3** | **P99 vs P99_9** | If findings move again at the next rung, the ladder has no stable regime |
| **4** | **23 months, not 1** | If Pittsburgh isn't always a source and Florida inflow is seasonal, neither is a finding |

## What Phase A established (and what's at risk)

V0 → P99_9 reversed six major findings — the Pittsburgh premium (2.35× → 0.95×), the
50–150 km hole (0.41 → 0.91), Cleveland's net position (+$100.8B → −$125.2M), the
absence of local supply chains (0.58% → 18.5% of dollars), triangle-closure lift
(8.2× → 1.5×), and the Houston trophic outlier (0.51 → 5.34). That rate of reversal
is the reason for this notebook.

What survives so far: the join (99.6%), closure (0.0% residual), coverage (98.4%),
the national shape of the book, and — the two candidate headlines —

1. money leaves the industrial Northeast/Midwest and lands in Florida and affluent suburbs
2. dollars per customer are flat across the country (~$29–37K per node in every ring)

Check 1 tests whether (1) is a TM story at all. Check 2 tests whether it's geographic.

In [ ]:
import json
import os
from datetime import datetime

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

## 0. Config

In [ ]:
PKG_METRICS_TABLE = "bdahd01p_dlcdi1_cdi_tm.cust_c2c_metrics"

WORK_DIR = None                      # same value used in Phase A
OUT_DIR_LOCAL = "../metrics/geo"

PKG_VERSION = "P99_9"                # primary. Check 3 also runs P99.
COMPARE_VERSION = "P99"
LATEST_TK = "2025-11"

MIN_UNIT_NODES = 25
TOP_N = 30

REPORT = {"generated_at": datetime.now().isoformat(timespec="seconds"), "phase": "A2"}


def note(k, v):
    REPORT[k] = v
    print(f"  {k}: {v}")


def show(df, n=TOP_N, truncate=False):
    df.show(n, truncate=truncate)


spark = (
    SparkSession.builder.appName("pkg_geo_phase_a2")
    .config("spark.sql.shuffle.partitions", "800")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .enableHiveSupport()
    .getOrCreate()
)
print("Spark", spark.version)

wtot = Window.partitionBy()

## 0.1 Load the Phase A node frame

`pkg_geo_nodes_{LATEST_TK}` was written at the end of Phase A. It already carries the
geo block joined to the P99_9 node metrics, so checks 1 and 2 need no re-derivation.

In [ ]:
nodes_path = f"{WORK_DIR}/pkg_geo_nodes_{LATEST_TK}" if WORK_DIR else None
if nodes_path is None:
    raise RuntimeError("Set WORK_DIR to the same value used in Phase A.")

nodes = spark.read.parquet(nodes_path).persist(StorageLevel.DISK_ONLY)
print(f"{nodes.count():,} nodes, {len(nodes.columns)} columns")
print(nodes.columns)

G = nodes.filter(F.col("geo_status") == "valid").persist(StorageLevel.DISK_ONLY)
note("A2_geo_valid_nodes", G.count())

## 1. Business vs consumer

The Florida inflow (Miami, Naples, Sarasota, Fort Myers, Clearwater, Orlando) plus
affluent-suburb sinks (Norristown, Lititz, Cherry Hill, Ellicott City, Rockville)
reads like retiree and wealth migration. If it is, it belongs to Retail and Wealth,
not Treasury Management.

### 1.1 Two independent classifiers

`naics_known` is the primary: a party carrying a NAICS code is a business.
`cust_name` entity tokens are the cross-check — independent evidence, so
disagreement between them is itself informative.

In [ ]:
name_u = F.upper(F.trim(F.coalesce(F.col("cust_name"), F.lit(""))))

ENTITY_RE = (
    r"(^| )(LLC|L\.L\.C|INC|INCORPORATED|CORP|CORPORATION|LTD|LIMITED|LP|LLP|LLLP|"
    r"PLLC|PC|PA|CO|COMPANY|TRUST|FOUNDATION|ASSN|ASSOCIATION|SOCIETY|CHURCH|"
    r"MINISTR|SCHOOL|ACADEMY|UNIVERSITY|COLLEGE|HOSPITAL|CLINIC|CENTER|CENTRE|"
    r"GROUP|HOLDINGS|PARTNERS|ENTERPRISES|INDUSTRIES|SERVICES|SOLUTIONS|SYSTEMS|"
    r"PROPERTIES|REALTY|CONSTRUCTION|CONTRACTING|RESTAURANT|MOTORS|AUTO|FARMS?|"
    r"MARKET|STORE|SHOP|SALON|DENTAL|MEDICAL|LAW|AGENCY|BANK|CREDIT UNION)( |$|\.|,)"
)

biz = (
    G
    .withColumn("naics_biz", F.when(F.col("naics_known") >= 1, 1).otherwise(0))
    .withColumn("name_biz", F.when(name_u.rlike(ENTITY_RE), 1).otherwise(0))
    .withColumn(
        "party_class",
        F.when((F.col("naics_biz") == 1) & (F.col("name_biz") == 1), "1_biz_both")
         .when(F.col("naics_biz") == 1, "2_biz_naics_only")
         .when(F.col("name_biz") == 1, "3_biz_name_only")
         .otherwise("4_consumer_like"),
    )
    .persist(StorageLevel.DISK_ONLY)
)

c1 = (
    biz.groupBy("party_class")
    .agg(
        F.count("*").alias("n_nodes"),
        F.round(F.sum("strength_sum") / 1e6, 1).alias("strength_musd"),
        F.round(F.expr("percentile_approx(strength, 0.5)"), 1).alias("med_strength"),
        F.round(F.avg("share_in_amt_individual"), 4).alias("avg_share_in_individual"),
        F.round(F.avg("hub_in_share"), 4).alias("avg_hub_in"),
        F.round(F.avg("months_active"), 1).alias("avg_months_active"),
    )
    .withColumn("pct_of_nodes", F.round(100 * F.col("n_nodes") / F.sum("n_nodes").over(wtot), 2))
    .withColumn("pct_of_strength",
                F.round(100 * F.col("strength_musd") / F.sum("strength_musd").over(wtot), 2))
    .orderBy("party_class")
)
show(c1)
REPORT["C1_party_class"] = [r.asDict() for r in c1.collect()]

**Reading 1.1.** `avg_share_in_individual` is the validity check on the classifier —
it is a graph-side measure of how much of a node's inflow comes from individuals, and
it should be markedly higher for `4_consumer_like` than for `1_biz_both`. If it isn't,
the classifier is not separating what it claims to.

The `2_` and `3_` disagreement classes matter: a NAICS code with no entity token is
often a sole proprietor; an entity token with no NAICS is an un-enriched business.

In [ ]:
# 1.2 — the decisive table. Metro net flow, business-only vs consumer-like.
BIZ = biz.filter(F.col("party_class").rlike("^[123]_"))
CON = biz.filter(F.col("party_class") == "4_consumer_like")


def metro_flow(df, label):
    return (
        df.groupBy("geo_unit", "u_state")
        .agg(
            F.count("*").alias(f"n_{label}"),
            F.sum("net_flow_sum").alias(f"_net_{label}"),
            F.sum("strength_sum").alias(f"_s_{label}"),
        )
        .withColumn(f"net_musd_{label}", F.round(F.col(f"_net_{label}") / 1e6, 1))
        .withColumn(f"strength_musd_{label}", F.round(F.col(f"_s_{label}") / 1e6, 1))
        .drop(f"_net_{label}", f"_s_{label}")
    )


flow_cmp = (
    metro_flow(BIZ, "biz")
    .join(metro_flow(CON, "con"), ["geo_unit", "u_state"], "outer")
    .fillna(0)
    .filter((F.col("n_biz") + F.col("n_con")) >= MIN_UNIT_NODES)
    .persist(StorageLevel.DISK_ONLY)
)

print("=== TOP NET SINKS — BUSINESS ONLY ===")
show(flow_cmp.orderBy(F.desc("net_musd_biz")).limit(25), 25, truncate=34)
print("=== TOP NET SINKS — CONSUMER-LIKE ONLY ===")
show(flow_cmp.orderBy(F.desc("net_musd_con")).limit(25), 25, truncate=34)
print("=== TOP NET SOURCES — BUSINESS ONLY ===")
show(flow_cmp.orderBy("net_musd_biz").limit(25), 25, truncate=34)

REPORT["C1_sinks_biz"] = [r.asDict() for r in flow_cmp.orderBy(F.desc("net_musd_biz")).limit(20).collect()]
REPORT["C1_sinks_con"] = [r.asDict() for r in flow_cmp.orderBy(F.desc("net_musd_con")).limit(20).collect()]
REPORT["C1_sources_biz"] = [r.asDict() for r in flow_cmp.orderBy("net_musd_biz").limit(20).collect()]

In [ ]:
# 1.3 — the Florida test, stated as a single number.
FL_SINKS = ["331", "342", "341", "328", "339", "337", "334", "322", "333", "335"]
NE_MW_SOURCES = ["191", "200", "402", "441", "452", "152", "070", "212", "480", "631"]


def bucket_share(df, label):
    tot = df.agg(F.sum("net_flow_sum")).collect()[0][0] or 0.0
    fl = df.filter(F.substring(F.col("geo_unit"), 1, 3).isin(FL_SINKS)) \
           .agg(F.sum("net_flow_sum")).collect()[0][0] or 0.0
    ne = df.filter(F.substring(F.col("geo_unit"), 1, 3).isin(NE_MW_SOURCES)) \
           .agg(F.sum("net_flow_sum")).collect()[0][0] or 0.0
    return {"population": label,
            "fl_net_musd": round(fl / 1e6, 1),
            "ne_mw_net_musd": round(ne / 1e6, 1),
            "all_net_musd": round(tot / 1e6, 1)}


fl_test = [bucket_share(BIZ, "business"), bucket_share(CON, "consumer_like"), bucket_share(G, "all")]
print(json.dumps(fl_test, indent=2))
REPORT["C1_florida_test"] = fl_test

**The verdict rule, written before seeing the numbers.**

- Florida inflow is **majority business** → it is a supply-chain / corporate-treasury
  story and stays with TM.
- Florida inflow is **majority consumer-like** → it is retiree and wealth migration.
  Real, interesting, and it belongs to Retail and Wealth. TM keeps only the business
  residual, which will be much smaller.
- **Both** → report them separately and never as one number.

## 2. NAICS shift-share — is it geography or industry mix?

Florida skews healthcare and retail; the Midwest skews manufacturing. Those industries
have different structural net-flow positions, so metro net flow could be entirely a
composition effect.

Standard decomposition, business nodes only, all edge-free because `naics2` is a node
column:

- **mix** — what the metro's net flow would be if every NAICS behaved at its national rate
- **local** — the residual: this metro's businesses behaving differently from their peers

Only `local` is a geographic finding.

In [ ]:
B = BIZ.filter(F.col("naics2").isNotNull() & (F.trim(F.col("naics2")) != ""))

# national net-flow rate per dollar of strength, by NAICS2
nat = (
    B.groupBy("naics2")
    .agg(F.sum("net_flow_sum").alias("_n"), F.sum("strength_sum").alias("_s"))
    .withColumn("nat_rate", F.col("_n") / F.col("_s"))
    .select("naics2", "nat_rate")
)
show(
    B.groupBy("naics2").agg(
        F.count("*").alias("n_nodes"),
        F.round(F.sum("net_flow_sum") / 1e6, 1).alias("net_musd"),
        F.round(F.sum("strength_sum") / 1e6, 1).alias("strength_musd"),
    ).join(F.broadcast(nat), "naics2")
     .withColumn("nat_rate", F.round("nat_rate", 4))
     .orderBy(F.desc("strength_musd")),
    30,
)

In [ ]:
ss = (
    B.groupBy("geo_unit", "u_state", "naics2")
    .agg(F.sum("net_flow_sum").alias("net"), F.sum("strength_sum").alias("s"))
    .join(F.broadcast(nat), "naics2", "left")
    .withColumn("expected", F.col("s") * F.col("nat_rate"))
    .groupBy("geo_unit", "u_state")
    .agg(
        F.sum("net").alias("_actual"),
        F.sum("expected").alias("_mix"),
        F.sum("s").alias("_s"),
        F.count("*").alias("n_naics2"),
    )
    .withColumn("_local", F.col("_actual") - F.col("_mix"))
    .withColumn("actual_musd", F.round(F.col("_actual") / 1e6, 1))
    .withColumn("mix_musd", F.round(F.col("_mix") / 1e6, 1))
    .withColumn("local_musd", F.round(F.col("_local") / 1e6, 1))
    .withColumn("strength_musd", F.round(F.col("_s") / 1e6, 1))
    .withColumn(
        "pct_explained_by_mix",
        F.round(100 * F.abs(F.col("_mix")) / F.greatest(F.abs(F.col("_actual")), F.lit(1.0)), 1),
    )
    .drop("_actual", "_mix", "_local", "_s")
    .filter(F.col("strength_musd") > 10)
    .persist(StorageLevel.DISK_ONLY)
)

print("=== metros where the LOCAL effect is largest (real geography) ===")
show(ss.orderBy(F.desc("local_musd")).limit(20), 20, truncate=34)
show(ss.orderBy("local_musd").limit(20), 20, truncate=34)
REPORT["C2_local_top"] = [r.asDict() for r in ss.orderBy(F.desc("local_musd")).limit(15).collect()]
REPORT["C2_local_bottom"] = [r.asDict() for r in ss.orderBy("local_musd").limit(15).collect()]

In [ ]:
# 2.1 — headline: how much of the observed metro net flow is mix, nationally?
agg = ss.agg(
    F.sum(F.abs(F.col("actual_musd"))).alias("sum_abs_actual"),
    F.sum(F.abs(F.col("mix_musd"))).alias("sum_abs_mix"),
    F.sum(F.abs(F.col("local_musd"))).alias("sum_abs_local"),
).collect()[0].asDict()
agg["pct_mix"] = round(100 * agg["sum_abs_mix"] / max(agg["sum_abs_actual"], 1), 1)
agg["pct_local"] = round(100 * agg["sum_abs_local"] / max(agg["sum_abs_actual"], 1), 1)
print(json.dumps(agg, indent=2))
REPORT["C2_decomposition"] = agg

In [ ]:
# 2.2 — the specific claim: Florida metros, decomposed
show(
    ss.filter(F.substring(F.col("geo_unit"), 1, 3).isin(FL_SINKS))
      .orderBy(F.desc("actual_musd")),
    20, truncate=34,
)

**Reading section 2.** If `pct_mix` is high (say >70%), the honest headline becomes
"healthcare and retail are net receivers, and Florida has more of them" — defensible,
much less interesting, and *not* a geography finding. If `pct_local` dominates, then
businesses in these metros genuinely behave differently from their national peers, and
the corridor work is worth the GPU pass.

## 3. Ablation robustness — P99 vs P99_9

V0 → P99_9 reversed six findings. If P99 → P99_9 moves things comparably, there is no
stable regime and every result stays provisional. If they agree, we have found it.

In [ ]:
pkg_raw = spark.table(PKG_METRICS_TABLE)
geo_lookup = nodes.select("node", "geo_unit", "u_state", "state", "km_from_pit", "geo_status")


def version_frame(ver):
    p = pkg_raw.filter((F.col("version") == F.lit(ver)) & (F.col("time_key") == F.lit(LATEST_TK)))
    return (
        p.select("node", "strength", "net_flow", "clustering_coef", "hub_in_share", "trophic_level")
         .join(geo_lookup, "node", "inner")
         .filter(F.col("geo_status") == "valid")
    )


rows = []
for ver in [COMPARE_VERSION, PKG_VERSION]:
    d = version_frame(ver)
    r = (
        d.withColumn(
            "ring",
            F.when(F.col("km_from_pit") < 50, "0_under_50km")
             .when(F.col("km_from_pit") < 150, "1_50_150km")
             .when(F.col("km_from_pit") < 400, "2_150_400km")
             .when(F.col("km_from_pit") < 1000, "3_400_1000km")
             .otherwise("4_over_1000km"),
        )
        .groupBy("ring")
        .agg(F.count("*").alias("n"), F.sum("strength").alias("s"))
        .withColumn("pct_n", 100 * F.col("n") / F.sum("n").over(wtot))
        .withColumn("pct_s", 100 * F.col("s") / F.sum("s").over(wtot))
        .withColumn("mult", F.round(F.col("pct_s") / F.col("pct_n"), 3))
        .withColumn("version", F.lit(ver))
        .select("version", "ring", "n", F.round("pct_n", 2).alias("pct_n"),
                F.round("pct_s", 2).alias("pct_s"), "mult")
    )
    rows.append(r)

ring_cmp = rows[0].unionByName(rows[1]).orderBy("ring", "version")
show(ring_cmp, 20)
REPORT["C3_ring_by_version"] = [r.asDict() for r in ring_cmp.collect()]

In [ ]:
# 3.1 — do the same metros lead the net-flow tables under both versions?
def metro_net(ver):
    return (
        version_frame(ver).groupBy("geo_unit")
        .agg(F.sum("net_flow").alias(f"net_{ver}"), F.count("*").alias(f"n_{ver}"))
        .filter(F.col(f"n_{ver}") >= MIN_UNIT_NODES)
    )


mn = (
    metro_net(COMPARE_VERSION).join(metro_net(PKG_VERSION), "geo_unit", "inner")
    .withColumn(f"musd_{COMPARE_VERSION}", F.round(F.col(f"net_{COMPARE_VERSION}") / 1e6, 1))
    .withColumn(f"musd_{PKG_VERSION}", F.round(F.col(f"net_{PKG_VERSION}") / 1e6, 1))
    .withColumn("sign_agree",
                F.when(F.signum(F.col(f"net_{COMPARE_VERSION}")) ==
                       F.signum(F.col(f"net_{PKG_VERSION}")), 1).otherwise(0))
)
note("C3_sign_agreement_pct", round(100 * (mn.agg(F.avg("sign_agree")).collect()[0][0] or 0), 2))
note("C3_rank_corr_spearman",
     round(mn.select(F.corr(F.col(f"musd_{COMPARE_VERSION}"), F.col(f"musd_{PKG_VERSION}"))).collect()[0][0] or 0, 4))
show(mn.orderBy(F.desc(f"musd_{PKG_VERSION}")).limit(20), 20, truncate=34)

**Reading section 3.** `sign_agreement_pct` is the number that matters. Cleveland
flipped sign between V0 and P99_9. If P99 and P99_9 agree on sign for the large
majority of metros, the direction of flow is a stable property and briefable. If not,
net flow is a threshold artifact and should not leave this notebook.

## 4. Temporal persistence — 23 months, not one

Every Phase A number is 2025-11. A finding that doesn't persist isn't one. Snowbird
seasonality would show up here as an unmistakable annual cycle in the Florida metros.

In [ ]:
pkg_all = pkg_raw.filter(F.col("version") == F.lit(PKG_VERSION))

panel = (
    pkg_all.select("node", "time_key", "strength", "net_flow")
    .join(geo_lookup.filter(F.col("geo_status") == "valid")
          .select("node", "geo_unit", "u_state", "km_from_pit"), "node", "inner")
    .persist(StorageLevel.DISK_ONLY)
)

# 4.1 — ring multiplier over time
ring_t = (
    panel.withColumn(
        "ring",
        F.when(F.col("km_from_pit") < 50, "0_under_50km")
         .when(F.col("km_from_pit") < 150, "1_50_150km")
         .when(F.col("km_from_pit") < 400, "2_150_400km")
         .when(F.col("km_from_pit") < 1000, "3_400_1000km")
         .otherwise("4_over_1000km"),
    )
    .groupBy("time_key", "ring")
    .agg(F.count("*").alias("n"), F.sum("strength").alias("s"))
    .withColumn("pct_n", 100 * F.col("n") / F.sum("n").over(Window.partitionBy("time_key")))
    .withColumn("pct_s", 100 * F.col("s") / F.sum("s").over(Window.partitionBy("time_key")))
    .withColumn("mult", F.round(F.col("pct_s") / F.col("pct_n"), 3))
    .groupBy("time_key").pivot("ring").agg(F.first("mult"))
    .orderBy("time_key")
)
show(ring_t, 30)
REPORT["C4_ring_multiplier_by_month"] = [r.asDict() for r in ring_t.collect()]

In [ ]:
# 4.2 — persistence of net position. A metro is a "persistent source/sink" only if it
# holds the same sign in a large majority of months. Same k-of-m discipline used for
# the edge backbone.
persist_tbl = (
    panel.groupBy("geo_unit", "u_state", "time_key")
    .agg(F.sum("net_flow").alias("net"), F.count("*").alias("n"))
    .filter(F.col("n") >= MIN_UNIT_NODES)
    .groupBy("geo_unit", "u_state")
    .agg(
        F.count("*").alias("n_months"),
        F.sum(F.when(F.col("net") < 0, 1).otherwise(0)).alias("months_source"),
        F.round(F.avg("net") / 1e6, 1).alias("avg_net_musd"),
        F.round(F.stddev("net") / 1e6, 1).alias("sd_net_musd"),
    )
    .withColumn("pct_months_source", F.round(100 * F.col("months_source") / F.col("n_months"), 1))
    .withColumn(
        "persistence",
        F.when(F.col("pct_months_source") >= 80, "persistent_SOURCE")
         .when(F.col("pct_months_source") <= 20, "persistent_SINK")
         .otherwise("unstable"),
    )
    .filter(F.col("n_months") >= 12)
    .persist(StorageLevel.DISK_ONLY)
)

show(persist_tbl.groupBy("persistence").agg(F.count("*").alias("n_metros")).orderBy("persistence"))
print("=== persistent SINKS, by average magnitude ===")
show(persist_tbl.filter(F.col("persistence") == "persistent_SINK")
     .orderBy(F.desc("avg_net_musd")).limit(20), 20, truncate=34)
print("=== persistent SOURCES ===")
show(persist_tbl.filter(F.col("persistence") == "persistent_SOURCE")
     .orderBy("avg_net_musd").limit(20), 20, truncate=34)
REPORT["C4_persistence"] = [r.asDict() for r in persist_tbl.groupBy("persistence").count().collect()]

In [ ]:
# 4.3 — seasonality in the Florida metros. A snowbird cycle is unmistakable here.
show(
    panel.filter(F.substring(F.col("geo_unit"), 1, 3).isin(FL_SINKS))
    .groupBy("time_key")
    .agg(F.round(F.sum("net_flow") / 1e6, 1).alias("fl_net_musd"),
         F.round(F.sum("strength") / 1e6, 1).alias("fl_strength_musd"))
    .orderBy("time_key"),
    30,
)

In [ ]:
# 4.4 — and the flat-dollars-per-node result, over time
show(
    panel.withColumn(
        "ring",
        F.when(F.col("km_from_pit") < 50, "0_under_50km")
         .when(F.col("km_from_pit") < 1000, "2_mid").otherwise("4_over_1000km"),
    )
    .groupBy("time_key", "ring")
    .agg(F.round(F.sum("strength") / F.count("*") / 1e3, 1).alias("strength_k_per_node"))
    .groupBy("time_key").pivot("ring").agg(F.first("strength_k_per_node"))
    .orderBy("time_key"),
    30,
)

## 5. Report

In [ ]:
os.makedirs(OUT_DIR_LOCAL, exist_ok=True)
if WORK_DIR:
    flow_cmp.write.mode("overwrite").parquet(f"{WORK_DIR}/pkg_geo_biz_vs_consumer_{LATEST_TK}")
    ss.write.mode("overwrite").parquet(f"{WORK_DIR}/pkg_geo_shiftshare_{LATEST_TK}")
    persist_tbl.write.mode("overwrite").parquet(f"{WORK_DIR}/pkg_geo_net_persistence")

print(json.dumps(REPORT, indent=2, default=str))
with open(f"{OUT_DIR_LOCAL}/phase_a2_report_{LATEST_TK}.json", "w") as fh:
    json.dump(REPORT, fh, indent=2, default=str)
print("report written")

---

## Decision rules, written in advance

Committing to these before seeing the output is the point — otherwise the numbers get
read to support whatever we already believe.

| Result | Consequence |
|---|---|
| Florida inflow majority **consumer** | Hand it to Retail/Wealth. TM keeps the business residual only. The metro net-flow headline comes out of the TM deck. |
| Florida inflow majority **business** | Stays a TM finding. Proceed to shift-share. |
| `pct_mix` **> 70%** | Reframe as an industry finding. Corridor/gravity work loses most of its value; scope the edge pass down. |
| `pct_local` **dominant** | Geography is real. Edge pass is justified — write `PKG_GEO_METRICS_SPEC.md` and schedule the GPU job. |
| P99 vs P99_9 sign agreement **< 80%** | Net flow is a threshold artifact. Do not brief it. |
| Ring multipliers **unstable across months** | The flat-dollars result is a single-month accident. Everything reverts to provisional. |
| Florida shows a clear **annual cycle** | It's seasonal migration, not a structural corridor. Report as seasonality. |

## What I need back

1. `C1_party_class` and `C1_florida_test` — the business/consumer split.
2. `C2_decomposition` — the single `pct_mix` number.
3. `C3_sign_agreement_pct`.
4. `C4_persistence` counts and the Florida monthly series.

## Deliberately not here

The CBSA crosswalk (external file, separate task) and the edge pass (waiting on
checks 2 and 4). Both are cheap to start once these four answers are in.